[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/differential_equations/08_odes_in_machine_learning/first_principles.ipynb)

# Topic 08: ODEs in Machine Learning

## 1. First-Principles Intuition & Motivation

### 1.1 Phenomenon: Learning Algorithms Are Sampled Flows

Almost every core object in modern machine learning is a discrete-time recursion:

- **Gradient descent**: $\theta_{k+1} = \theta_k - \eta \nabla L(\theta_k)$
- **Residual layer**: $h_{l+1} = h_l + f(h_l, \theta_l)$
- **Recurrent cell**: $h_{t+1} = \phi(W h_t + U x_t)$
- **Diffusion denoising step**: $x_{k-1} = x_k + (\text{drift and score terms})$

Each of these is exactly one step of a numerical integrator applied to some ordinary differential equation.
Gradient descent is forward Euler applied to the **gradient flow** $\dot{\theta} = -\nabla L(\theta)$;
a ResNet layer is forward Euler with step $\Delta t = 1$ applied to a feature flow $\dot{h} = f(h, \theta)$;
diffusion samplers integrate a reverse-time differential equation.
The discrete algorithm inherits its qualitative behavior — convergence, oscillation, blow-up, invertibility — from the continuous flow it samples.

### 1.2 Goal: Analyze the Flow, Then Discretize

The strategy of this module is a two-step reduction:

1. **Continuum limit**: identify the ODE whose discretization is the given algorithm. The ODE is usually far easier to analyze: convergence proofs become Lyapunov arguments, expressiveness questions become topology of flows, and gradient computations become adjoint (variational) equations.
2. **Discretization audit**: return to the discrete algorithm and quantify what the finite step changes — stability ceilings on the learning rate, discretization error of a ResNet, solver tolerance effects on Neural ODE gradients.

This mirrors the classical numerical-analysis pipeline of Topics 01–07, but run in reverse: instead of discretizing a known ODE, we are handed the discretization and must reconstruct the flow.

### 1.3 Assumptions, Variables, and Notation

- $L : \mathbb{R}^p \to \mathbb{R}$: a differentiable training loss with parameters $\theta \in \mathbb{R}^p$; $L^{\ast} = \inf L$ is attained.
- $F(h, t, \theta)$: a neural vector field, Lipschitz in $h$ (true for standard architectures with bounded weights and Lipschitz activations such as $\tanh$ or ReLU), so Picard–Lindelöf (Topic 02) applies.
- $h(t) \in \mathbb{R}^d$: the feature state of a Neural ODE; $h(0)$ is the input, $h(T)$ feeds the loss $\ell(h(T))$.
- $a(t) = \partial \ell / \partial h(t) \in \mathbb{R}^d$: the adjoint state.
- $J(t) = \partial F / \partial h$ evaluated along the trajectory: the local Jacobian of the dynamics.
- $p_t$: the probability density of the state at time $t$ when the initial state is random.
- $\eta$: learning rate; $\gamma$: friction/damping coefficient; $\beta(t)$: diffusion noise schedule; $\Delta$: discretization step of a state-space model.

Throughout, $\lVert \cdot \rVert$ denotes the Euclidean norm and $\operatorname{tr}$ the matrix trace.

## 2. Rigorous Mathematical Definitions & Theorem Statements

### 2.1 Gradient Flow and the Polyak–Łojasiewicz Condition

**Definition (Gradient flow).** The gradient flow of a differentiable loss $L$ is the autonomous ODE

$$
\dot{\theta}(t) = -\nabla L(\theta(t)), \qquad \theta(0) = \theta_0
$$

Gradient descent with learning rate $\eta$ is its forward-Euler discretization $\theta_{k+1} = \theta_k - \eta \nabla L(\theta_k)$.

**Definition (PL inequality).** $L$ satisfies the Polyak–Łojasiewicz condition with constant $\mu \gt 0$ if for all $\theta$

$$
\frac{1}{2} \lVert \nabla L(\theta) \rVert^2 \ge \mu \left( L(\theta) - L^{\ast} \right)
$$

Every $\mu$-strongly-convex function is PL, but PL also holds for some non-convex losses (e.g., sufficiently overparameterized networks near interpolation).

**Theorem (Exponential convergence).** Under PL, the gradient flow satisfies $L(\theta(t)) - L^{\ast} \le e^{-2\mu t} \left( L(\theta_0) - L^{\ast} \right)$. (Proof in Section 3.1.)

### 2.2 Momentum: Heavy-Ball and Nesterov ODEs

**Definition (Heavy-ball ODE).** Momentum gradient methods are discretizations of the second-order dynamics

$$
\ddot{\theta}(t) + \gamma \dot{\theta}(t) + \nabla L(\theta(t)) = 0
$$

a particle of unit mass rolling on the loss surface with friction $\gamma$ — precisely the damped oscillator of Topic 03 when $L$ is quadratic.

**Theorem (Nesterov ODE, Su–Boyd–Candès 2016).** Nesterov's accelerated gradient method with step $\eta \to 0$ converges to the trajectory of

$$
\ddot{X}(t) + \frac{3}{t} \dot{X}(t) + \nabla f(X(t)) = 0, \qquad X(0) = x_0, \quad \dot{X}(0) = 0
$$

and for convex $f$ every solution satisfies

$$
f(X(t)) - f^{\ast} \le \frac{2 \lVert x_0 - x^{\ast} \rVert^2}{t^2}
$$

The vanishing damping $3/t$ is the continuous-time signature of acceleration: strong friction early (stability), weak friction late (speed).

### 2.3 Neural ODEs and the Adjoint State

**Definition (Neural ODE, Chen et al. 2018).** Given a parameterized vector field $F(h, t, \theta)$, the Neural ODE layer maps input $h_0$ to $h(T)$, where

$$
\frac{dh}{dt} = F(h(t), t, \theta), \qquad h(0) = h_0
$$

The forward pass is an ODE solve; the "depth" is the integration horizon $T$, and the computational depth is the number of function evaluations (NFE) chosen by the solver.

**Theorem (Adjoint sensitivity).** Let $\ell(h(T))$ be a differentiable loss on the terminal state. Define the adjoint $a(t) = \partial \ell / \partial h(t)$. Then

$$
\frac{da}{dt} = -\left( \frac{\partial F}{\partial h} \right)^{T} a(t), \qquad a(T) = \frac{\partial \ell}{\partial h(T)}
$$

and the parameter gradient is

$$
\frac{d\ell}{d\theta} = \int_0^T a(t)^{T} \frac{\partial F}{\partial \theta}(h(t), t, \theta) \, dt
$$

Both integrals run backward from $t = T$ to $t = 0$ alongside a re-integration of $h$, giving gradients with memory cost independent of NFE. (Proof in Section 3.3.)

### 2.4 Continuous Normalizing Flows

**Definition (CNF).** Push a base random variable $h(0) \sim p_0$ through the flow of $\dot{h} = F(h, t, \theta)$. The terminal density $p_T$ is the model distribution.

**Theorem (Instantaneous change of variables).** If $F$ is Lipschitz with Jacobian $J(t) = \partial F / \partial h$ along the trajectory $h(t)$, then the log-density along the trajectory obeys the scalar ODE

$$
\frac{d}{dt} \log p_t(h(t)) = -\operatorname{tr} \big( J(t) \big)
$$

so exact log-likelihoods require only a trace — $O(d)$ per evaluation with a Hutchinson estimator $\operatorname{tr}(J) = \mathbb{E}_{v}\left[ v^{T} J v \right]$ for random probe vectors $v$ with identity covariance — instead of an $O(d^3)$ determinant. (Proof in Section 3.4.)

This is the continuous limit of the discrete change-of-variables formula $\log p_Y(y) = \log p_X(x) - \log \lvert \det \partial y / \partial x \rvert$ used by discrete normalizing flows.

### 2.5 Diffusion Models: OU Noising and the Probability Flow ODE

**Definition (Variance-preserving forward process).** Diffusion models corrupt data with an Ornstein–Uhlenbeck process; with constant schedule $\beta$,

$$
dx = -\tfrac{1}{2} \beta x \, dt + \sqrt{\beta} \, dW_t
$$

Its mean $m(t)$ and variance $v(t)$ obey deterministic ODEs (derived in Section 3.5): $m' = -\tfrac{\beta}{2} m$ and $v' = \beta (1 - v)$, so any initial law is driven toward the stationary $\mathcal{N}(0, 1)$.

**Theorem (Probability flow ODE, Song et al. 2021).** The deterministic ODE

$$
\frac{dx}{dt} = -\tfrac{1}{2} \beta x - \tfrac{1}{2} \beta \, \nabla_x \log p_t(x)
$$

has exactly the same time marginals $p_t$ as the forward SDE. Replacing $\nabla_x \log p_t$ by a learned score network and integrating this ODE backward in time is deterministic sampling; DDIM is a particular discretization of it.

**Definition (State-space sequence model).** A linear SSM is $\dot{x} = A x + B u$, $y = C x$; sequence architectures such as S4 and Mamba use the zero-order-hold discretization $x_{k+1} = e^{A\Delta} x_k + A^{-1}\left( e^{A\Delta} - I \right) B u_k$, i.e., structured matrix exponentials from Topic 04.

## 3. Step-by-Step Mathematical Proofs & Derivations

### 3.1 Proof: Exponential Convergence of Gradient Flow under PL

**Claim.** If $L$ satisfies the PL inequality with constant $\mu$, then along the gradient flow $L(\theta(t)) - L^{\ast} \le e^{-2\mu t}\left( L(\theta_0) - L^{\ast} \right)$.

**Step 1 — Energy dissipation identity.** Differentiate the suboptimality $E(t) = L(\theta(t)) - L^{\ast}$ along the flow using the chain rule:

$$
\frac{dE}{dt} = \nabla L(\theta)^{T} \dot{\theta} = \nabla L(\theta)^{T} \left( -\nabla L(\theta) \right) = -\lVert \nabla L(\theta) \rVert^2
$$

The loss is non-increasing along every gradient flow trajectory — this identity alone proves monotone descent.

**Step 2 — Apply PL.** The PL inequality gives $\lVert \nabla L \rVert^2 \ge 2\mu E$, hence

$$
\frac{dE}{dt} \le -2\mu E(t)
$$

**Step 3 — Grönwall.** By the differential form of Grönwall's inequality (Topic 02), $E(t) \le E(0) e^{-2\mu t}$.

Result:

$$
\boxed{L(\theta(t)) - L^{\ast} \le e^{-2\mu t} \left( L(\theta_0) - L^{\ast} \right)}
$$

Gradient descent inherits the discrete analogue $E_{k+1} \le (1 - \eta\mu(2 - \eta L_{s}))\,E_k$ only for $\eta$ below the stability ceiling set by the smoothness constant $L_{s}$ — the Euler stability condition in disguise.

### 3.2 Proof: Critical Damping of the Heavy-Ball ODE on a Quadratic Mode

**Claim.** For a quadratic loss mode $L(\theta) = \tfrac{1}{2}\lambda\theta^2$ with curvature $\lambda \gt 0$, the heavy-ball ODE $\ddot{\theta} + \gamma\dot{\theta} + \lambda\theta = 0$ decays fastest when $\gamma = 2\sqrt{\lambda}$.

**Step 1 — Characteristic roots.** Substituting $\theta = e^{rt}$ (Topic 03) yields $r^2 + \gamma r + \lambda = 0$, so

$$
r_{\pm} = \frac{-\gamma \pm \sqrt{\gamma^2 - 4\lambda}}{2}
$$

**Step 2 — Underdamped regime ($\gamma \lt 2\sqrt{\lambda}$).** The roots are complex with $\operatorname{Re}(r_{\pm}) = -\gamma/2$: the envelope decays like $e^{-\gamma t/2}$, improving as $\gamma$ grows, at the price of oscillation.

**Step 3 — Overdamped regime ($\gamma \gt 2\sqrt{\lambda}$).** Both roots are real and the slow root governs the decay:

$$
r_{\text{slow}} = \frac{-\gamma + \sqrt{\gamma^2 - 4\lambda}}{2} = -\frac{2\lambda}{\gamma + \sqrt{\gamma^2 - 4\lambda}} \longrightarrow -\frac{\lambda}{\gamma} \quad (\gamma \to \infty)
$$

so excessive friction makes convergence arbitrarily slow.

**Step 4 — Optimum at the boundary.** The decay rate $\min\left( \gamma/2, \; -r_{\text{slow}} \right)$ increases in $\gamma$ on the underdamped side and decreases on the overdamped side, so it is maximized exactly at $\gamma^{\ast} = 2\sqrt{\lambda}$, where $r_{+} = r_{-} = -\sqrt{\lambda}$ and $\theta(t) = (c_1 + c_2 t)e^{-\sqrt{\lambda} t}$.

Result:

$$
\boxed{\gamma^{\ast} = 2\sqrt{\lambda}, \qquad \text{optimal decay } e^{-\sqrt{\lambda}\, t}}
$$

For a full Hessian spectrum $\lambda \in [\mu, L_{s}]$, one damping must serve all modes; choosing $\gamma = 2\sqrt{\mu}$ protects the slowest mode and yields the $\sqrt{\mu}$ rate that becomes the $\sqrt{\mu/L_{s}}$ acceleration factor after discretization.

### 3.3 Proof: The Adjoint Sensitivity Method

**Claim.** For $\dot{h} = F(h, t, \theta)$ with loss $\ell(h(T))$, the adjoint $a(t) = \partial\ell/\partial h(t)$ satisfies $\dot{a} = -(\partial F/\partial h)^{T} a$ and $d\ell/d\theta = \int_0^T a^{T} (\partial F/\partial\theta) \, dt$.

**Step 1 — Composition across an infinitesimal step.** The state at $t + \varepsilon$ is $h(t + \varepsilon) = h(t) + \varepsilon F(h(t), t, \theta) + O(\varepsilon^2)$, a differentiable map of $h(t)$. By the chain rule, sensitivities compose:

$$
a(t)^{T} = a(t + \varepsilon)^{T} \frac{\partial h(t + \varepsilon)}{\partial h(t)} = a(t + \varepsilon)^{T}\left( I + \varepsilon \frac{\partial F}{\partial h} + O(\varepsilon^2) \right)
$$

**Step 2 — Differentiate.** Rearranging and letting $\varepsilon \to 0$:

$$
\frac{da^{T}}{dt} = \lim_{\varepsilon \to 0} \frac{a(t + \varepsilon)^{T} - a(t)^{T}}{\varepsilon} = -a(t)^{T} \frac{\partial F}{\partial h}
$$

which is the stated linear ODE $\dot{a} = -(\partial F/\partial h)^{T} a$ — the transpose of the variational equation governing forward perturbations $\delta h$, integrated in reverse. This is literally backpropagation through an infinitesimally deep network.

**Step 3 — Parameter gradient by state augmentation.** Treat $\theta$ as an extra state with dynamics $\dot{\theta} = 0$. The augmented adjoint $a_{\theta}(t) = \partial\ell/\partial\theta(t)$ obeys, by the same argument applied to the augmented Jacobian,

$$
\dot{a}_{\theta} = -\left( \frac{\partial F}{\partial \theta} \right)^{T} a(t), \qquad a_{\theta}(T) = 0
$$

**Step 4 — Integrate backward.** Solving from $T$ down to $0$:

$$
a_{\theta}(0) = \frac{d\ell}{d\theta} = \int_0^T a(t)^{T} \frac{\partial F}{\partial \theta}\, dt
$$

Result:

$$
\boxed{\dot{a} = -\left( \frac{\partial F}{\partial h} \right)^{T} a, \qquad \frac{d\ell}{d\theta} = \int_0^T a(t)^{T} \frac{\partial F}{\partial \theta}\, dt}
$$

All quantities on the right are computed by one reverse-time ODE solve of the triple $(h, a, a_{\theta})$, so memory does not grow with the number of forward solver steps.

### 3.4 Proof: Instantaneous Change of Variables (Trace Formula)

**Claim.** Along a trajectory of $\dot{h} = F(h, t)$, the log-density satisfies $\frac{d}{dt}\log p_t(h(t)) = -\operatorname{tr}(\partial F/\partial h)$.

**Step 1 — Flow Jacobian dynamics.** Let $\Phi_t$ be the flow map $h(0) \mapsto h(t)$ and $J_{\Phi}(t) = \partial \Phi_t / \partial h(0)$. Differentiating the ODE with respect to the initial condition (the variational equation, justified by smooth dependence from Topic 02):

$$
\frac{d}{dt} J_{\Phi}(t) = \frac{\partial F}{\partial h}\big( h(t), t \big) \, J_{\Phi}(t), \qquad J_{\Phi}(0) = I
$$

**Step 2 — Jacobi's formula (Liouville).** For any differentiable matrix function with $\dot{M} = A(t) M$ and $M(0) = I$, the determinant satisfies

$$
\frac{d}{dt} \det M(t) = \det M(t) \, \operatorname{tr}\big( M^{-1} \dot{M} \big) = \det M(t)\, \operatorname{tr}\big( A(t) \big)
$$

Applied to $J_{\Phi}$: $\frac{d}{dt} \log \det J_{\Phi}(t) = \operatorname{tr}\big( \partial F/\partial h \big)$. In particular $\det J_{\Phi}(t) \gt 0$ for all $t$ (it starts at 1 and can never cross 0), so the flow is orientation-preserving.

**Step 3 — Conservation of probability.** The change-of-variables formula for the pushforward density states $p_t(h(t)) \, \det J_{\Phi}(t) = p_0(h(0))$, a constant in $t$. Taking logarithms and differentiating:

$$
\frac{d}{dt} \log p_t(h(t)) = -\frac{d}{dt} \log \det J_{\Phi}(t) = -\operatorname{tr}\left( \frac{\partial F}{\partial h} \right)
$$

Result:

$$
\boxed{\frac{d}{dt} \log p_t(h(t)) = -\operatorname{tr}\left( \frac{\partial F}{\partial h}(h(t), t) \right)}
$$

A CNF therefore augments the state with one scalar, integrates $\big( \dot{h}, \dot{\ell} \big) = \big( F, -\operatorname{tr}(\partial F/\partial h) \big)$, and reads off exact log-likelihoods at $t = T$.

### 3.5 Proof: Ornstein–Uhlenbeck Moment ODEs and the Stationary Law

**Claim.** For the VP forward process $dx = -\tfrac{1}{2}\beta x\, dt + \sqrt{\beta}\, dW_t$ with $x(0)$ of mean $m_0$ and variance $v_0$, the mean and variance obey $m' = -\tfrac{\beta}{2} m$ and $v' = \beta(1 - v)$, so $p_t \to \mathcal{N}(0, 1)$.

**Step 1 — Mean ODE.** Taking expectations of the integral form of the SDE, the martingale noise term has zero mean, leaving the deterministic linear ODE

$$
\frac{dm}{dt} = -\frac{\beta}{2} m(t) \quad \Longrightarrow \quad m(t) = m_0 e^{-\beta t/2}
$$

**Step 2 — Second-moment ODE.** Itô's formula applied to $x^2$ gives $d(x^2) = \left( -\beta x^2 + \beta \right) dt + 2x\sqrt{\beta}\, dW_t$; taking expectations, $s(t) = \mathbb{E}\left[ x^2 \right]$ satisfies $s' = -\beta s + \beta$.

**Step 3 — Variance ODE.** With $v = s - m^2$ and $\left( m^2 \right)' = -\beta m^2$:

$$
v' = s' - \left( m^2 \right)' = -\beta s + \beta + \beta m^2 = \beta\left( 1 - v \right)
$$

This is a linear first-order ODE (Topic 01); by the integrating factor $e^{\beta t}$,

$$
v(t) = 1 + \left( v_0 - 1 \right) e^{-\beta t}
$$

**Step 4 — Stationary limit.** As $t \to \infty$, $m(t) \to 0$ and $v(t) \to 1$ exponentially fast, and since a linear SDE with Gaussian (or deterministic) initial condition stays Gaussian:

$$
\boxed{m(t) = m_0 e^{-\beta t/2}, \qquad v(t) = 1 + (v_0 - 1) e^{-\beta t}, \qquad p_t \to \mathcal{N}(0, 1)}
$$

This is why the diffusion forward process may be truncated at finite $T$: the data law is already exponentially close to the tractable prior.

### 3.6 Proof: Neural ODE Flows Are Homeomorphisms

**Claim.** If $F(h, t)$ is continuous in $t$ and Lipschitz in $h$, the time-$T$ flow map $\Phi_T : h(0) \mapsto h(T)$ is a homeomorphism of $\mathbb{R}^d$ (continuous bijection with continuous inverse).

**Step 1 — Well-defined and injective.** By Picard–Lindelöf (Topic 02) each initial condition generates a unique trajectory. If $\Phi_T(u) = \Phi_T(w)$, run the ODE backward in time from that common terminal point: reversed time $s = T - t$ gives $\dot{g} = -F(g, T - s)$, again Lipschitz, so the backward solution is also unique, forcing $u = w$.

**Step 2 — Surjective.** For any target $y \in \mathbb{R}^d$, solve the backward equation from $y$ for time $T$ to obtain a state $u$ with $\Phi_T(u) = y$. Hence $\Phi_T^{-1}$ exists and is itself a flow map.

**Step 3 — Continuity both ways.** For two solutions $h_u, h_w$ started at $u, w$, subtract the integral equations and apply Grönwall's inequality with Lipschitz constant $K$:

$$
\lVert h_u(t) - h_w(t) \rVert \le \lVert u - w \rVert \, e^{K t}
$$

so $\Phi_T$ is (Lipschitz-)continuous; the same bound for the reversed field gives continuity of $\Phi_T^{-1}$.

Result:

$$
\boxed{\Phi_T \text{ is a homeomorphism; Neural ODE features can be deformed but never torn, glued, or reflected across trajectories}}
$$

**Consequence (expressiveness limit).** In $d = 1$, homeomorphisms obtained from flows preserve order, so no 1D Neural ODE represents $x \mapsto -x$; more generally flows cannot change orientation or linking of sets. **Augmented Neural ODEs** fix this by lifting to $\mathbb{R}^{d + q}$, where the extra dimensions let trajectories pass around each other.

## 4. Computational & Algorithmic Insights

### 4.1 Solvers, NFE, and Tolerances

A Neural ODE's computational depth is the **number of function evaluations (NFE)** its adaptive solver spends — typically Dormand–Prince RK45 with absolute/relative tolerances. Key practical facts:

- NFE grows as the learned dynamics stiffen during training; regularizers that penalize $\lVert F \rVert$ or the Jacobian (kinetic energy, STEER, RNODE) keep dynamics cheap to integrate.
- Solver tolerance is a *model hyperparameter*: gradients from the adjoint method are exact for the continuous flow but both forward and backward passes are only tolerance-accurate; loose tolerances inject bias into training.
- The reverse-time reconstruction of $h(t)$ in the adjoint method can drift from the stored forward pass when the dynamics are unstable in reverse; algebraically reversible or checkpointed solvers restore agreement.

```text
ADJOINT BACKWARD PASS (Chen et al. 2018)
  input: terminal state h(T), gradient a(T) = dL/dh(T)
  s(T) = [ h(T), a(T), 0 ]
  integrate backward from T to 0 the augmented system:
      dh/dt      = F(h, t, theta)
      da/dt      = - (dF/dh)^T a          # vector-Jacobian product
      da_th/dt   = - (dF/dtheta)^T a      # vector-Jacobian product
  output: a(0) = dL/dh(0), and dL/dtheta = a_th(0) with flipped sign convention
```

Each right-hand side needs one VJP — the same primitive as ordinary backprop — so the cost per backward step matches a backprop step; the saving is memory: $O(1)$ in NFE instead of $O(\text{NFE})$.

### 4.2 Discretize-then-Optimize vs Optimize-then-Discretize

Two inequivalent ways to get gradients of an ODE-based model:

| Strategy | What is differentiated | Memory | Gradient exactness |
|---|---|---|---|
| Discretize-then-optimize (backprop through solver) | The actual discrete computation graph | Grows with NFE | Exact for the discrete model |
| Optimize-then-discretize (adjoint method) | The continuous flow, then discretize the adjoint ODE | Constant in NFE | Exact only as solver error tends to 0 |

The two commute in the limit of vanishing step size; at finite tolerance they differ, and the mismatch can destabilize training. A robust default: adjoint for memory-bound problems with smooth dynamics, backprop-through-solver (with checkpointing) when gradients must match the deployed discrete computation exactly.

**Stability of forward propagation.** For deep ResNets viewed as Euler schemes, forward stability requires the eigenvalues of $I + \Delta t \, \partial f/\partial h$ to stay near the unit circle. Haber & Ruthotto derive well-posed architectures by constraining $\partial f/\partial h$ toward skew-symmetry (eigenvalues near the imaginary axis) — features neither explode nor die, the architectural analogue of choosing dynamics with $\operatorname{Re}(\lambda) \approx 0$.

### 4.3 Gradient Propagation as an ODE Stability Problem

Backpropagation through time is the adjoint equation of the forward recurrence. For an RNN or Neural ODE, the gradient norm evolves according to the linearized (variational) dynamics:

$$
\lVert a(0) \rVert \approx \lVert a(T) \rVert \cdot \exp\left( \int_0^T \operatorname{Re}\,\lambda_{\text{eff}}(t) \, dt \right)
$$

where $\lambda_{\text{eff}}$ tracks the (transposed) Jacobian's exponents along the trajectory — the **Lyapunov exponents** of the learned dynamics:

- All exponents strongly negative: gradients vanish exponentially in horizon length (the RNN forgets).
- Any exponent positive: gradients explode; training oscillates or diverges.
- Exponents near zero: long-range credit assignment survives — the design target.

This single picture explains three fixes: **orthogonal/unitary RNNs** pin Jacobian singular values to 1; **LSTM/GRU gating** implements a leaky-integrator ODE $\dot{h} = -h/\tau + (\text{input})$ whose effective $\tau$ is learned per unit (forget gate $f \approx e^{-\Delta/\tau}$), letting the network place its own timescales; **state-space models** (S4, Mamba) parameterize $A$ with controlled spectra (e.g., HiPPO matrices) and evaluate $e^{A\Delta}$ exactly, as in Topic 04, so stability is guaranteed by construction rather than learned by luck.

## 5. Real-World Physics & AI/ML Applications

### 5.1 Optimization and Training Dynamics

- **Learning-rate schedules as time reparameterizations.** A schedule $\eta(t)$ turns gradient descent into a non-autonomous flow $\dot{\theta} = -\eta(t)\nabla L$; warmup and decay are changes of time variable in the gradient flow.
- **Momentum tuning by damping analysis.** Given Hessian curvature estimates $[\mu, L_{s}]$, the heavy-ball analysis of Section 3.2 prescribes near-critical damping for the slowest mode — the ODE-level explanation of why momentum $\approx 0.9$ works across so many problems.
- **Edge of stability.** Training with large $\eta$ operates beyond the Euler stability ceiling of the sharpest Hessian mode; observed oscillations in the loss are exactly the numerical instability of Topic 01's stiff-equation analysis, not a bug in the optimizer.
- **Physics parallel:** the heavy-ball ODE is the equation of a massive particle in a potential with linear drag — the same equation as an RLC circuit or damped spring (Topic 03), so intuition transfers verbatim.

### 5.2 Generative Modeling

- **Continuous normalizing flows / FFJORD** use the trace formula (Section 3.4) for maximum-likelihood training of free-form dynamics; Hutchinson probes make the trace stochastic but unbiased.
- **Diffusion models** train a score network on the OU-corrupted data (Section 3.5); sampling integrates either the reverse SDE (stochastic, e.g., ancestral/DDPM) or the probability flow ODE (deterministic, e.g., DDIM, higher-order samplers like DPM-Solver). The ODE view enables few-step sampling, exact likelihoods, and semantically meaningful latent interpolation, because the map noise-to-image is an invertible flow (Section 3.6).
- **Flow matching / rectified flows** learn the vector field of an ODE that transports noise to data along nearly straight paths — regression on $F$ directly, no SDE simulation needed.

### 5.3 Sequence Models, Science, and Control

- **Latent ODEs and ODE-RNNs** (Rubanova et al. 2019) handle irregularly-sampled clinical and astronomical time series: between observations the latent state evolves by an ODE; at observations it is updated — a continuous-discrete filter.
- **State-space models** (S4, Mamba) achieve long-context sequence modeling by exact ZOH discretization of structured linear ODEs; their convolution-kernel view $K(t) = C e^{At} B$ links Topic 04 (matrix exponentials) with Topic 06 (transfer functions).
- **Physics-informed learning:** PINNs and neural operators embed ODE/PDE residuals in the loss (Topic 07), while Hamiltonian and Lagrangian neural networks hard-code symplectic structure so learned dynamics conserve energy.
- **Control and RL:** continuous-time value functions satisfy the Hamilton–Jacobi–Bellman equation; discount factors $e^{-\rho t}$ act as the Laplace weight of Topic 06.

## 6. Canonical Literature Mapping & References

| Concept in this module | Canonical source | Where |
|---|---|---|
| Gradient flow, PL convergence | Polyak (1963); Karimi, Nutini & Schmidt (2016) | PL inequality analysis |
| Heavy-ball ODE | Polyak (1964); Topic 03 of this curriculum | Damped oscillator theory |
| Nesterov ODE and Lyapunov proof | Su, Boyd & Candès (2016), JMLR 17(153) | Sections 2–3 |
| Neural ODEs, adjoint method | Chen, Rubanova, Bettencourt & Duvenaud (2018), NeurIPS | Sections 2, 4, 5 |
| CNFs and trace estimation | Grathwohl et al. (2019), FFJORD, ICLR | Section 4 |
| Expressiveness of flows, augmentation | Dupont, Doucet & Teh (2019), Augmented Neural ODEs, NeurIPS | Section 3 |
| Diffusion SDEs, probability flow ODE | Song et al. (2021), ICLR; Ho, Jain & Abbeel (2020) | Sections 3–4 |
| Stable deep architectures | Haber & Ruthotto (2017), Inverse Problems 34(1) | Section 3 |
| Dynamical-systems view of deep learning | E, W. (2017), Commun. Math. Stat. 5(1) | Whole-paper perspective |
| State-space sequence models | Gu, Goel & Ré (2022), S4, ICLR; Gu & Dao (2023), Mamba | Sections 2–3 |
| Latent ODEs for time series | Rubanova, Chen & Duvenaud (2019), NeurIPS | Section 3 |
| Flow/stability foundations used here | Arnold, *Ordinary Differential Equations*; Hirsch, Smale & Devaney | Chapters on flows, linear systems, stability |

**Curriculum cross-references.** Existence-uniqueness and Grönwall: [Topic 02](../02_existence_uniqueness_picard_lindelof/README.md). Damped oscillators: [Topic 03](../03_second_order_linear_odes/README.md). Matrix exponentials and variational equations: [Topic 04](../04_systems_of_odes_matrix_exponential/README.md). Stability and Lyapunov functions: [Topic 05](../05_phase_plane_and_stability_analysis/README.md). Transfer functions and Laplace weights: [Topic 06](../06_laplace_transform_methods/README.md). Survey-level companion: [calculus Topic 15](../../calculus/15_ordinary_differential_equations/README.md).